# Introduction to Python

**Topic:** `1.1` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Boot banner formatter

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** variables, f-strings, functions

**Explanation and reasoning**

The function validates its input before formatting, so an empty name fails loudly instead of producing a misleading banner. The f-string assembles the whole line in one expression, which keeps the embedded quotes readable.

In [ ]:
def boot_banner(name: str, version: str) -> str:
    if not name:
        raise ValueError("name must be non-empty")
    return f"ROBO-X '{name}' ready on Python {version}"

**Complexity**

Time O(n) in the length of the strings; space O(n) for the result.

**Edge cases and failure modes**

An empty name raises ValueError; spaces inside the name survive intact.

**Alternative approaches**

'...'.format(...) or string concatenation are equivalent here.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert boot_banner("rover-01", "3.12.3") == "ROBO-X 'rover-01' ready on Python 3.12.3"

---

## Solution 2 — Sensor string to float

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** type conversion, exception handling, defaults

**Explanation and reasoning**

float() raises ValueError for text it cannot parse, and catching that specific exception keeps genuine bugs such as TypeError visible. The default argument supplies the fallback without a second parameter.

In [ ]:
def reading(raw, default: float = 0.0) -> float:
    try:
        return float(raw)
    except ValueError:
        return default

**Complexity**

Time O(n) in the length of the input string; space O(1).

**Edge cases and failure modes**

'n/a' and '' both fall back to the default; surrounding whitespace is tolerated.

**Alternative approaches**

str.strip() plus an explicit numeric check is longer but clearer to beginners.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert reading('12.5') == 12.5
assert reading('n/a') == 0.0
assert reading('n/a', default=-1.0) == -1.0

---

## Solution 3 — Total distance travelled

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** loops, accumulators, floats

**Explanation and reasoning**

sum() runs in a single C-level pass, which is both the clearest and the fastest option here. round() at the boundary keeps float accumulation error out of the returned value, and an empty list sums to 0 naturally.

In [ ]:
def total_distance(segments) -> float:
    return round(sum(segments), 2)

**Complexity**

Time O(n); space O(1) beyond the input.

**Edge cases and failure modes**

An empty list returns 0.0 because the sum of nothing is 0.

**Alternative approaches**

An explicit for-loop with a running total makes the accumulation visible for teaching.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert total_distance([1.5, 2.0, 3.25]) == 6.75
assert total_distance([]) == 0.0

---

## Solution 4 — Clamp a battery percentage

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** conditionals, comparison, clamping

**Explanation and reasoning**

Two early-return guards make the clamp explicit: each bound is checked and returned immediately, so the happy path is the fall-through. That is easier to read and to debug than nested conditionals.

In [ ]:
def clamp_pct(value):
    if value < 0:
        return 0
    if value > 100:
        return 100
    return value

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

Exactly 0 and exactly 100 pass through unchanged.

**Alternative approaches**

max(0, min(100, value)) is shorter but hides the two distinct rules.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert clamp_pct(118.0) == 100
assert clamp_pct(-5) == 0
assert clamp_pct(42) == 42

---

## Solution 5 — Pair sensor names with readings

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** dictionaries, zip, iteration

**Explanation and reasoning**

zip() yields pairs lazily and stops at the shorter sequence, which is exactly the required truncation behaviour, so no length check is needed. dict() then consumes those pairs as key/value entries.

In [ ]:
def pairs(names, readings):
    return dict(zip(names, readings))

**Complexity**

Time O(n); space O(n) for the resulting dict.

**Edge cases and failure modes**

Duplicate names collapse, keeping the last reading; empty inputs give an empty dict.

**Alternative approaches**

A for-loop over range(min(len(a), len(b))) states the truncation rule explicitly.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert pairs(['imu', 'batt'], [0.3, 12.6]) == {'imu': 0.3, 'batt': 12.6}
assert pairs([], []) == {}
assert pairs(['a', 'b'], [1]) == {'a': 1}

---

## Solution 6 — Parse a configuration line

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** strings, dictionaries, validation

**Explanation and reasoning**

_convert() tries the narrowest type first, then float, and finally falls back to the original string, so the ordering drives correctness. partition('=') splits on the first separator only, which keeps values containing '=' intact, and blank segments are skipped before validation.

In [ ]:
def _convert(token: str):
    try:
        return int(token)
    except ValueError:
        pass
    try:
        return float(token)
    except ValueError:
        return token


def parse_config(line: str) -> dict:
    config: dict = {}
    for chunk in line.split(';'):
        chunk = chunk.strip()
        if not chunk:
            continue
        if '=' not in chunk:
            raise ValueError(f'malformed pair: {chunk!r}')
        key, _, value = chunk.partition('=')
        config[key.strip()] = _convert(value.strip())
    return config

**Complexity**

Time O(n) in the line length; space O(k) for k parsed pairs.

**Edge cases and failure modes**

Trailing ';' is skipped; whitespace is stripped; a missing '=' raises ValueError.

**Alternative approaches**

A regular expression with named groups is concise but hides the parsing steps.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert parse_config('max_speed=1.5; mode=auto') == {'max_speed': 1.5, 'mode': 'auto'}
assert parse_config('battery=48') == {'battery': 48}

---

## Solution 7 — Distance between two waypoints

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** tuples, arithmetic, math

**Explanation and reasoning**

math.hypot computes sqrt(dx*dx + dy*dy) while avoiding intermediate overflow for very large coordinates, so it is safer than squaring by hand. Destructuring the two sequences first keeps the arithmetic tied to the geometry rather than the container type.

In [ ]:
import math


def distance(p, q) -> float:
    dx = q[0] - p[0]
    dy = q[1] - p[1]
    return round(math.hypot(dx, dy), 3)

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

Identical points give 0.0; negative coordinates are handled without special cases.

**Alternative approaches**

math.dist(p, q) does the same thing in one call from Python 3.8 onwards.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert distance((0, 0), (3, 4)) == 5.0
assert distance((1.5, -2.0), (1.5, -2.0)) == 0.0

---

## Solution 8 — Run-length encode a string

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** strings, loops, state

**Explanation and reasoning**

The encoder keeps the current character and its run length as state, and flushes the chunk whenever the character changes. The final flush after the loop is required, otherwise the last run is silently dropped. Joining at the end avoids quadratic string concatenation.

In [ ]:
def rle(text: str) -> str:
    if not text:
        return ''
    parts = []
    current = text[0]
    count = 1
    for ch in text[1:]:
        if ch == current:
            count += 1
        else:
            parts.append(f'{current}{count}')
            current, count = ch, 1
    parts.append(f'{current}{count}')
    return ''.join(parts)

**Complexity**

Time O(n) in one pass; space O(n) in the number of runs.

**Edge cases and failure modes**

Empty input returns ''; a single-character string returns that character with count 1.

**Alternative approaches**

itertools.groupby produces the same encoding more compactly.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert rle('aaabbc') == 'a3b2c1'
assert rle('') == ''
assert rle('zzzz') == 'z4'

---

## Solution 9 — Word frequency table

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** strings, dictionaries, sorting

**Explanation and reasoning**

Each token is lower-cased and stripped of non-alphabetic characters, which normalises punctuation without needing a regex. The sort key negates the count so the most frequent word comes first, with the word itself breaking ties alphabetically for deterministic output.

In [ ]:
def word_counts(text: str) -> dict:
    counts: dict = {}
    for token in text.split():
        word = ''.join(ch for ch in token.lower() if ch.isalpha())
        if word:
            counts[word] = counts.get(word, 0) + 1
    ordered = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    return dict(ordered)

**Complexity**

Time O(n log k) for k distinct words; space O(k).

**Edge cases and failure modes**

Empty or punctuation-only input returns an empty dict.

**Alternative approaches**

collections.Counter is shorter; the explicit loop shows the counting step for teaching.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert word_counts('Dock dock; rover!') == {'dock': 2, 'rover': 1}
assert word_counts('') == {}

---

## Solution 10 — Validate robot state records

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dictionaries, validation, error reporting

**Explanation and reasoning**

Each record is checked with an early continue, so a failure records one precise reason and moves on rather than aborting the batch. The bool check comes first because bool is a subclass of int in Python and would otherwise pass the numeric test.

In [ ]:
def validate_states(records):
    valid, errors = [], {}
    for index, record in enumerate(records):
        if not isinstance(record, dict):
            errors[index] = 'record is not a mapping'
            continue
        if 'id' not in record:
            errors[index] = 'missing id'
            continue
        pct = record.get('battery_pct')
        if pct is None:
            errors[index] = 'missing battery_pct'
            continue
        if isinstance(pct, bool) or not isinstance(pct, (int, float)):
            errors[index] = 'battery_pct is not numeric'
            continue
        if not 0 <= pct <= 100:
            errors[index] = 'battery_pct out of range'
            continue
        valid.append(record)
    return valid, errors

**Complexity**

Time O(n) in the number of records; space O(n) worst case.

**Edge cases and failure modes**

An empty list gives ([], {}); non-dict entries are reported by index.

**Alternative approaches**

A dataclass with __post_init__ validation pushes the checks into the type.

**Tests — run the cell to verify the reference solution**

In [ ]:
valid, errors = validate_states([{'id': 'r1', 'battery_pct': 50}, {'id': 'r2'}])
assert valid == [{'id': 'r1', 'battery_pct': 50}]
assert errors == {1: 'missing battery_pct'}
assert validate_states([]) == ([], {})

---

# Robotics challenge solution — ROBO-X Challenge: Telemetry Readiness Controller

**Explanation**

The controller reads each requested channel and catches SensorError so a single failed sensor degrades the result instead of crashing the telemetry loop. The summary field is derived from the readings, and the ok flag reports whether every channel produced a value.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot, SensorError  # noqa: E402

DEFAULT_CHANNELS = ('distance', 'temperature', 'battery_voltage')


def process_telemetry(data=None) -> dict:
    channels = list(data) if data else list(DEFAULT_CHANNELS)
    robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
    readings = {}
    for channel in channels:
        try:
            readings[channel] = robot.read_sensor(channel)
        except SensorError:
            readings[channel] = None
    state = robot.status()
    return {
        'readings': readings,
        'battery_pct': state['battery_pct'],
        'ok': all(v is not None for v in readings.values()),
    }

**Complexity**

Time O(c) for c channels; space O(c).

**Notes and trade-offs**

An unknown channel raises SensorError and is recorded as None; an empty request uses the defaults.

In [ ]:
result = process_telemetry([])
assert result['ok'] is True
assert result['battery_pct'] > 0

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.